# 🧙‍♂️ Qwen 3B - Chat avec Persona Personnalisé

Ce notebook permet d'utiliser le modèle **Qwen2.5-3B-Instruct** avec un système de persona défini via un fichier Markdown.

## Fonctionnalités
- 📁 Upload d'un fichier `.md` pour définir le persona de l'IA
- 💬 Interface de chat interactive
- 🎛️ Paramètres de génération ajustables
- 💾 Export de la conversation

---
**Prérequis** : GPU T4 ou supérieur (gratuit sur Colab)

## 1️⃣ Installation des dépendances

In [ ]:
%%capture
!pip install -q transformers>=4.37.0 accelerate>=0.26.0 bitsandbytes>=0.42.0
!pip install -q torch>=2.1.0
!pip install -q gradio>=4.0.0
!pip install -q markdown

print("✅ Dépendances installées !")

## 2️⃣ Configuration et imports

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import files
import markdown
import json
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Vérification GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮 GPU détecté : {gpu_name}")
    print(f"💾 VRAM : {gpu_memory:.1f} GB")
else:
    print("⚠️ Aucun GPU détecté ! Allez dans Runtime > Change runtime type > T4 GPU")

# Configuration globale
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DEFAULT_PERSONA = """Tu es un assistant IA serviable et amical. Tu réponds de manière claire et concise."""

print(f"\n📦 Modèle configuré : {MODEL_ID}")

## 3️⃣ Chargement du Persona (fichier Markdown)

In [ ]:
class PersonaManager:
    def __init__(self):
        self.persona_text = DEFAULT_PERSONA
        self.persona_name = "Assistant par défaut"
        self.persona_file = None

    def load_from_markdown(self, content: str, filename: str = "persona.md"):
        """Charge un persona depuis le contenu d'un fichier Markdown."""
        self.persona_text = content.strip()
        self.persona_file = filename

        # Essayer d'extraire le nom du persona depuis un titre H1
        lines = content.split('\n')
        for line in lines:
            if line.startswith('# '):
                self.persona_name = line[2:].strip()
                break
        else:
            self.persona_name = filename.replace('.md', '')

        print(f"✅ Persona chargé : {self.persona_name}")
        print(f"📄 Fichier : {filename}")
        print(f"📏 Longueur : {len(self.persona_text)} caractères")

    def get_system_prompt(self) -> str:
        """Retourne le prompt système formaté."""
        return self.persona_text

    def preview(self):
        """Affiche un aperçu du persona."""
        preview_text = self.persona_text[:500]
        if len(self.persona_text) > 500:
            preview_text += "\n\n[... tronqué ...]"
        display(HTML(f"""
        <div style="background: #1a1a2e; color: #eee; padding: 15px; border-radius: 10px; border-left: 4px solid #7c3aed;">
            <h3 style="color: #a78bfa; margin-top: 0;">🎭 {self.persona_name}</h3>
            <pre style="white-space: pre-wrap; font-size: 12px;">{preview_text}</pre>
        </div>
        """))

# Instance globale
persona_manager = PersonaManager()
print("📋 PersonaManager initialisé avec le persona par défaut")

In [ ]:
# 📁 UPLOAD DE VOTRE FICHIER MARKDOWN
# Exécutez cette cellule pour uploader votre fichier persona.md

print("📤 Sélectionnez votre fichier Markdown (.md) pour le persona...")
print("   (Cliquez sur 'Choisir un fichier' ci-dessous)\n")

try:
    uploaded = files.upload()

    if uploaded:
        for filename, content in uploaded.items():
            if filename.endswith('.md'):
                text_content = content.decode('utf-8')
                persona_manager.load_from_markdown(text_content, filename)
                print("\n" + "="*50)
                persona_manager.preview()
            else:
                print(f"⚠️ Le fichier {filename} n'est pas un fichier Markdown (.md)")
    else:
        print("ℹ️ Aucun fichier uploadé. Utilisation du persona par défaut.")
        persona_manager.preview()

except Exception as e:
    print(f"ℹ️ Upload annulé ou erreur : {e}")
    print("   Utilisation du persona par défaut.")
    persona_manager.preview()

## 4️⃣ Chargement du modèle Qwen 3B

In [ ]:
%%time

print(f"🔄 Chargement de {MODEL_ID}...")
print("   (Cela peut prendre 2-5 minutes la première fois)\n")

# Configuration de quantification pour économiser la VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

# Chargement du modèle
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

# Informations sur le modèle
print(f"\n✅ Modèle chargé avec succès !")
print(f"📊 Paramètres : ~3B (quantifié en 4-bit)")
print(f"💾 VRAM utilisée : {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 5️⃣ Moteur de Chat

In [ ]:
class ChatEngine:
    def __init__(self, model, tokenizer, persona_manager):
        self.model = model
        self.tokenizer = tokenizer
        self.persona_manager = persona_manager
        self.conversation_history = []
        self.generation_config = {
            "max_new_tokens": 512,
            "temperature": 0.7,
            "top_p": 0.9,
            "top_k": 50,
            "repetition_penalty": 1.1,
            "do_sample": True
        }

    def reset_conversation(self):
        """Réinitialise l'historique de conversation."""
        self.conversation_history = []
        print("🔄 Conversation réinitialisée")

    def set_generation_params(self, **kwargs):
        """Met à jour les paramètres de génération."""
        self.generation_config.update(kwargs)
        print(f"⚙️ Paramètres mis à jour : {kwargs}")

    def generate_response(self, user_message: str) -> str:
        """Génère une réponse à partir du message utilisateur."""

        # Ajouter le message utilisateur à l'historique
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })

        # Construire les messages avec le système prompt
        messages = [
            {"role": "system", "content": self.persona_manager.get_system_prompt()}
        ] + self.conversation_history

        # Tokenisation avec le template de chat
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        # Génération
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.generation_config["max_new_tokens"],
                temperature=self.generation_config["temperature"],
                top_p=self.generation_config["top_p"],
                top_k=self.generation_config["top_k"],
                repetition_penalty=self.generation_config["repetition_penalty"],
                do_sample=self.generation_config["do_sample"],
                pad_token_id=self.tokenizer.eos_token_id
            )

        # Décodage de la réponse
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        # Ajouter la réponse à l'historique
        self.conversation_history.append({
            "role": "assistant",
            "content": response
        })

        return response

    def export_conversation(self) -> str:
        """Exporte la conversation au format JSON."""
        export_data = {
            "persona": self.persona_manager.persona_name,
            "timestamp": datetime.now().isoformat(),
            "messages": self.conversation_history,
            "generation_config": self.generation_config
        }
        return json.dumps(export_data, ensure_ascii=False, indent=2)

# Initialisation du moteur de chat
chat_engine = ChatEngine(model, tokenizer, persona_manager)
print("💬 Moteur de chat initialisé !")
print(f"🎭 Persona actif : {persona_manager.persona_name}")

## 6️⃣ Interface de Chat Interactive

In [ ]:
def display_message(role: str, content: str):
    """Affiche un message formaté."""
    if role == "user":
        bg_color = "#2d3748"
        icon = "👤"
        align = "right"
        border_color = "#4299e1"
    else:
        bg_color = "#1a202c"
        icon = "🤖"
        align = "left"
        border_color = "#48bb78"

    display(HTML(f"""
    <div style="text-align: {align}; margin: 10px 0;">
        <div style="display: inline-block; max-width: 80%; background: {bg_color};
                    color: #fff; padding: 12px 16px; border-radius: 12px;
                    border-left: 3px solid {border_color}; text-align: left;">
            <span style="font-size: 14px;">{icon} <b>{role.capitalize()}</b></span>
            <p style="margin: 8px 0 0 0; white-space: pre-wrap;">{content}</p>
        </div>
    </div>
    """))

def chat_interface():
    """Interface de chat simple avec widgets."""

    # Widgets
    input_box = widgets.Textarea(
        placeholder='Écrivez votre message ici...',
        layout=widgets.Layout(width='100%', height='80px')
    )

    send_btn = widgets.Button(
        description='📤 Envoyer',
        button_style='primary',
        layout=widgets.Layout(width='120px')
    )

    reset_btn = widgets.Button(
        description='🔄 Reset',
        button_style='warning',
        layout=widgets.Layout(width='100px')
    )

    export_btn = widgets.Button(
        description='💾 Export',
        button_style='info',
        layout=widgets.Layout(width='100px')
    )

    output_area = widgets.Output(
        layout=widgets.Layout(width='100%', min_height='300px', border='1px solid #333')
    )

    status_label = widgets.HTML(value="<i>Prêt à discuter...</i>")

    # Handlers
    def on_send(b):
        user_msg = input_box.value.strip()
        if not user_msg:
            return

        input_box.value = ""
        status_label.value = "<i>⏳ Génération en cours...</i>"

        with output_area:
            display_message("user", user_msg)

        try:
            response = chat_engine.generate_response(user_msg)
            with output_area:
                display_message("assistant", response)
            status_label.value = "<i>✅ Réponse générée</i>"
        except Exception as e:
            status_label.value = f"<i>❌ Erreur : {str(e)}</i>"

    def on_reset(b):
        chat_engine.reset_conversation()
        output_area.clear_output()
        status_label.value = "<i>🔄 Conversation réinitialisée</i>"

    def on_export(b):
        export_json = chat_engine.export_conversation()
        filename = f"chat_export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(export_json)
        files.download(filename)
        status_label.value = f"<i>💾 Exporté : {filename}</i>"

    send_btn.on_click(on_send)
    reset_btn.on_click(on_reset)
    export_btn.on_click(on_export)

    # Layout
    header = widgets.HTML(f"""
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                padding: 15px; border-radius: 10px; margin-bottom: 10px;">
        <h2 style="color: white; margin: 0;">🧙‍♂️ Chat avec {persona_manager.persona_name}</h2>
        <p style="color: #ddd; margin: 5px 0 0 0; font-size: 12px;">Modèle : Qwen2.5-3B-Instruct</p>
    </div>
    """)

    button_box = widgets.HBox([send_btn, reset_btn, export_btn])

    display(widgets.VBox([
        header,
        output_area,
        input_box,
        button_box,
        status_label
    ]))

# Lancer l'interface
chat_interface()

## 7️⃣ Mode Chat Simplifié (Alternative)

Si l'interface avec widgets ne fonctionne pas bien, utilisez cette version simplifiée :

In [ ]:
# 💬 CHAT SIMPLE - Exécutez cette cellule plusieurs fois pour chatter

# Entrez votre message ici :
USER_MESSAGE = "Bonjour ! Qui es-tu ?"  # @param {type:"string"}

if USER_MESSAGE.strip():
    print(f"👤 Vous : {USER_MESSAGE}")
    print("\n⏳ Génération en cours...\n")

    response = chat_engine.generate_response(USER_MESSAGE)

    print(f"🤖 {persona_manager.persona_name} :")
    print("-" * 40)
    print(response)
    print("-" * 40)
else:
    print("ℹ️ Entrez un message dans USER_MESSAGE ci-dessus")

## 8️⃣ Paramètres de Génération

In [ ]:
# @title ⚙️ Ajuster les paramètres de génération
max_new_tokens = 512  # @param {type:"slider", min:64, max:2048, step:64}
temperature = 0.7  # @param {type:"slider", min:0.1, max:2.0, step:0.1}
top_p = 0.9  # @param {type:"slider", min:0.1, max:1.0, step:0.05}
top_k = 50  # @param {type:"slider", min:1, max:100, step:1}
repetition_penalty = 1.1  # @param {type:"slider", min:1.0, max:2.0, step:0.05}

chat_engine.set_generation_params(
    max_new_tokens=max_new_tokens,
    temperature=temperature,
    top_p=top_p,
    top_k=top_k,
    repetition_penalty=repetition_penalty
)

print("\n📊 Configuration actuelle :")
for k, v in chat_engine.generation_config.items():
    print(f"   • {k}: {v}")

## 9️⃣ Utilitaires

In [ ]:
# 🔄 Réinitialiser la conversation (conserver le persona)
chat_engine.reset_conversation()

In [ ]:
# 📤 Charger un nouveau persona
print("Uploadez un nouveau fichier .md pour changer de persona...")
uploaded = files.upload()

for filename, content in uploaded.items():
    if filename.endswith('.md'):
        persona_manager.load_from_markdown(content.decode('utf-8'), filename)
        chat_engine.reset_conversation()
        persona_manager.preview()

In [ ]:
# 💾 Exporter la conversation
export_json = chat_engine.export_conversation()
filename = f"conversation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

with open(filename, 'w', encoding='utf-8') as f:
    f.write(export_json)

files.download(filename)
print(f"✅ Conversation exportée : {filename}")

In [ ]:
# 📊 Informations sur la mémoire GPU
print(f"💾 VRAM utilisée : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"💾 VRAM réservée : {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(f"💾 VRAM totale : {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

---

## 📝 Format du fichier Persona (.md)

Votre fichier Markdown peut suivre ce format :

```markdown
# Nom du Persona

Tu es [description du personnage].

## Personnalité
- Trait 1
- Trait 2

## Instructions
- Toujours répondre en [langue]
- Utiliser [style de communication]

## Contexte
[Informations supplémentaires sur l'univers, le lore, etc.]
```

Le titre H1 (`#`) sera utilisé comme nom du persona dans l'interface.